# Test Baseline

In [32]:
import mastershelf.recipes.preprocessing
data = mastershelf.recipes.preprocessing.load_recipes()

/home/mitri/.pyenv/versions/master_shelf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The file exists. Loading ...


In [133]:
from mastershelf.inventory.inventory import *

In [63]:
path = "../../raw_data/sam-test2.jpg"
path2 = "../../raw_data/sam-test.jpg"

In [ ]:
from mastershelf.api.fast import *


In [ ]:


def get_photo_inv():
    response = None
    try:
        response = set(mastershelf.api.fast.app.ingredients(path2))
        return response
    except:
        return PHOTO_INV

In [36]:
def get_user_inv():
    inventory = None
    # streamlit case à cocher
    if inventory:
        return inventory
    return  USER_INV

In [ ]:
user_inventory = get_user_inv()
photo_ingredient = get_photo_inv()


In [134]:
available_names = list(set(set(user_inventory.keys()) | set(photo_ingredient["ingredients_list"])))

In [135]:
available_names

['chicken breast',
 'egg',
 'soy sauce',
 'cheddar cheese',
 'olive oil',
 'tomato',
 'bell pepper',
 'paprika',
 'salt',
 'garlic',
 'onion',
 'curry powder',
 'carrot',
 'rice',
 'black pepper',
 'mustard',
 'pasta']

In [40]:
data.head()

,name,ingredients_raw,steps,servings,persons,portion_size,type_dish,type_diet,type_meal,type_occasion,type_origin,ingredients
0,Cherry Streusel Cobbler,"[2 (21 ounce) cans cherry pie filling, 2 ...","[Preheat oven to 375°F., Spread cherry pie fil...",6.0,1,347 g,[cobblers-and-crisps],[],[desserts],[],[north-american],"[cherry pie filling, condense milk, margarine,..."
1,Reuben and Swiss Casserole Bake,"[1/2-1 lb corned beef, cooked and chopped...","[Set oven to 350 degrees F., Butter a 9 x 13-i...",4.0,1,207 g,[],[dietary],[main-dish],[],[],"[corn beef, sauerkraut cold water, swiss chees..."
2,Tropical Orange Layer Cake,"[1 (18 ounce) pkge.orange cake mix, 1 (3 ...","[In a large mixing bowl, combine the first 6 i...",16.0,1,191 g,[cakes],"[dietary, low-protein]",[desserts],[],[],"[orange cake mix, instant vanilla pudding, ora..."
3,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,"[1/2 cup butter, room temperature , 1/2 c...","[Cream butter and sugars together., Blend in m...",24.0,1,26 g,[cookies-and-brownies],[],[desserts],[],[],"[butter, brown sugar, granulate sugar, milk, v..."
4,Teriyaki Pork Chops,"[1 (16 ounce) bottle teriyaki sauce, 4 ...",[I like to marinade them overnight in a ziploc...,4.0,1,313 g,[pork-chops],"[dietary, high-protein, low-carb]",[main-dish],[],[north-american],"[teriyaki sauce, pork chop]"


In [106]:
top_recipes = data.copy()

In [107]:
def recipe_coverage(recipe_ingredients, user_ingredients):
    if len(recipe_ingredients) == 0:
        return 0

    matches = 0

    for recipe_ing in recipe_ingredients:
        found = any(
            ingredient_match(user_ing, recipe_ing)
            for user_ing in user_ingredients
        )

        if found:
            matches += 1

    return matches / len(recipe_ingredients)

In [108]:
import re
def ingredient_match(user_ing, recipe_ing):
    user_ing = user_ing.lower().strip()
    recipe_ing = recipe_ing.lower().strip()

    return (
        re.search(rf"\b{re.escape(user_ing)}\b", recipe_ing)
        is not None
    )

In [112]:
top_recipes["coverage"] = top_recipes["ingredients"].apply(lambda x: recipe_coverage(x, available_names))

In [113]:
top_recipes["n_ingredients"] = top_recipes["ingredients"].apply(len)

possible_recipes = (
    top_recipes[top_recipes["coverage"] == 1.0]
    .sort_values(
        by=["coverage", "n_ingredients"],
        ascending=[False, False]
    )
)

In [115]:
possible_recipes[["name","steps","coverage", "n_ingredients"]].head(10)

,name,steps,coverage,n_ingredients
211201,"Chicken, Squash, &amp; Sun-Dried Tomato Alfred...","[First, broil the chicken (4-6 minutes each si...",1.0,8
4783,Green and Red Tomato Rice,"[Add oil to a medium size sauce pan., Add chop...",1.0,7
177915,Tri Color Pepper Salad,[Heat oil in a large skillet over med- high he...,1.0,7
202659,Cauliflower Vegetable Fried Rice,[Heat large skillet or wok over medium heat an...,1.0,7
266474,"Peperonata ( Peppers , Tomatoes , Onions)",[Cut bell pepper in half and clean the inside....,1.0,7
3549,Mexican Pasta (Sopa?),"[Heat the olive oil in a 2 qt sauce pan., Put ...",1.0,6
11857,Fancy Stir Fry,[1st cook the chicken thoroughly in a large fr...,1.0,6
27776,Roasted Red Peppers with Yellow Pepper Puree,[Keep the peppers whole and roast over an open...,1.0,6
28269,Roasted Veggies,"[Preheat oven to 375 degrees., Mix all ingredi...",1.0,6
62731,My Rice Dish,"[Preheat large skillet with oil, add rice and ...",1.0,6


In [129]:
verif = possible_recipes.iloc[2].ingredients
verif

['olive oil vegetable oil',
 'clove garlic',
 'onion',
 'red bell pepper',
 'green bell pepper',
 'yellow bell pepper',
 'plum tomato']

In [128]:
possible_recipes.iloc[2].steps

['Heat oil in a large skillet over med- high heat.',
 'Cook garlic and onion in oil, stirring often, until tender.',
 'Reduce heat to med-low and add peppers.',
 'Cover and cook 5 minutes.',
 'Add tomatoes, salt and pepper and stir until combined.',
 'Cook uncovered about 5 minutes, stirring often, until peppers are crisp tender.',
 'Serve warm or cover and refrigerate at least 2 hours.']

In [140]:
possible_recipes.iloc[9].ingredients_raw

['3   cups    cooked rice',
 '2   tablespoons    olive oil',
 '1   small    onion, chopped ',
 '1   cup    frozen peas and carrot',
 '2 -3   tablespoons    soy sauce',
 '2       eggs, lightly beaten ',
 '2   tablespoons    green onions, chopped ']

In [142]:
possible_recipes.iloc[9]

name                                                    My Rice Dish
ingredients_raw    [3   cups    cooked rice, 2   tablespoons    o...
steps              [Preheat large skillet with oil, add rice and ...
servings                                                         4.0
persons                                                            1
portion_size                                                   235 g
type_dish                                                         []
type_diet                      [dietary, healthy, low-saturated-fat]
type_meal                                              [side-dishes]
type_occasion                                                     []
type_origin                                                       []
ingredients        [rice, olive oil, onion, peas carrot, soy sauc...
coverage                                                         1.0
n_ingredients                                                      6
Name: 62731, dtype: object

In [2]:
df.head()

NameError: name 'df' is not defined